# Colab smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/X-Lemon-X/project_ai/blob/main/notebooks/colab_smoke_test.ipynb)

Checks that the training side of the pipeline ([docs/proposals/0006-pipeline.md](../docs/proposals/0006-pipeline.md))
works in Colab **before we have a real dataset**:

1. GPU and PyTorch
2. Google Drive mount
3. Cloning the repo
4. A **synthetic** dataset in the planned format (one `.npz` per episode)
5. Zip → Drive → copy back → unzip (the recommended Drive workflow)
6. Fine-tuning a pretrained ResNet18 on it
7. Checkpoint to Drive and resume
8. TorchScript export and reload

The synthetic images are simple drawings (sky, grass, road, one box), not
Isaac Sim renders. They only exist to exercise the code path.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
The whole notebook takes a few minutes.

In [ ]:
# Settings
USE_DRIVE = True            # False: keep everything in /content (lost when the VM is deleted)
REPO_URL = "https://github.com/X-Lemon-X/project_ai.git"
BRANCH = "main"             # set to a feature branch to test it before merging
N_EPISODES = 60             # synthetic episodes
STEPS = 50                  # steps per episode
H, W = 120, 160             # low-resolution camera_front, as planned
EPOCHS = 3

checks = {}                 # name -> (ok, detail), printed at the end

## 1. GPU and PyTorch

In [ ]:
import time, torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 2**30
    a = torch.randn(4096, 4096, device=DEVICE)
    torch.cuda.synchronize(); t = time.time()
    for _ in range(10):
        a @ a
    torch.cuda.synchronize()
    tflops = 10 * 2 * 4096**3 / (time.time() - t) / 1e12
    detail = f"{name}, {mem:.1f} GiB, ~{tflops:.1f} TFLOP/s fp32"
    checks["gpu"] = (True, detail)
else:
    checks["gpu"] = (False, "no GPU - Runtime > Change runtime type > T4 GPU")
print(checks["gpu"])

## 2. Google Drive

In [ ]:
from pathlib import Path
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/project_ai")
        checks["drive"] = (True, str(BASE))
    except Exception as e:  # not in Colab, or mount refused
        BASE = Path("/content/project_ai_local") if Path("/content").exists() else Path("project_ai_local")
        checks["drive"] = (False, f"{type(e).__name__}: {e} - using {BASE}")
else:
    BASE = Path("/content/project_ai_local") if Path("/content").exists() else Path("project_ai_local")
    checks["drive"] = (True, f"skipped (USE_DRIVE=False), using {BASE}")
RUN_DIR = BASE / "runs" / "smoke_test"
RUN_DIR.mkdir(parents=True, exist_ok=True)
(BASE / "datasets").mkdir(parents=True, exist_ok=True)
print(checks["drive"])

## 3. Clone the repo

In [ ]:
import subprocess
WORK = Path("/content") if Path("/content").exists() else Path(".")
REPO = WORK / "project_ai"
if not REPO.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
ok = (REPO / "scene_gen.py").exists()
commit = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True).stdout.strip()
checks["clone"] = (ok, f"{BRANCH}: {commit}" if ok else "scene_gen.py not found")
print(checks["clone"])

## 4. Synthetic dataset

Same idea as the real generator: a point mass drives along +X with fixed
heading, an expert picks the action, and with probability 0.2 a random action
is executed instead (DART-style perturbation). The **label is always the
expert action**. Each episode is saved as one compressed `.npz` with the same
fields we plan for real data.

Actions: 0 forward, 1 backward, 2 left (+y), 3 right (−y).

In [ ]:
import numpy as np

ACTIONS = ["forward", "backward", "left", "right"]
STEP = 0.2                      # m per step
ROAD_HALF, SHOULDER = 2.0, 1.5  # drivable |y| <= 3.5
R_ROBOT = 0.75


def render(rng, lat, obstacles, style):
    """Crude front view: sky, grass, road trapezoid, boxes. Image x to the right = world -y."""
    img = np.empty((H, W, 3), np.uint8)
    hz = 50                                                  # horizon row
    img[:hz] = style["sky"]
    img[hz:] = style["grass"]
    rows = np.arange(hz, H)
    t = (rows - hz + 1) / (H - hz)                           # 0 far .. 1 near
    px_per_m = 35 * t
    cx = W / 2 + lat * px_per_m                              # road centre (world y=0) in image
    half = ROAD_HALF * px_per_m
    cols = np.arange(W)[None, :]
    road = np.abs(cols - cx[:, None]) <= half[:, None]
    img[hz:][road] = style["road"]
    centre_line = np.abs(cols - cx[:, None]) <= 0.06 * px_per_m[:, None] + 0.5
    if style["dashed"]:
        centre_line &= ((rows // 6) % 2 == 0)[:, None]
    img[hz:][centre_line] = (240, 240, 240)
    for dx, oy, r, colour in sorted(obstacles, key=lambda o: -o[0]):  # far first
        if dx <= 0.5:
            continue
        tt = min(1.0, 1.5 / dx)
        s = 35 * tt
        x0 = int(W / 2 - (oy - lat) * s - r * s)
        x1 = int(W / 2 - (oy - lat) * s + r * s)
        yb = int(hz + (H - hz) * tt)
        yt = int(yb - 1.6 * r * s)
        img[max(yt, 0):max(min(yb, H), 0), max(x0, 0):max(min(x1, W), 0)] = colour
    noise = rng.integers(-6, 7, img.shape)
    return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)


def expert(lat, obstacles, lane_y):
    """Rule expert for the smoke test (the real one will be the cost-to-go grid)."""
    for dx, oy, r, _ in sorted(obstacles, key=lambda o: o[0]):
        if 0 < dx < 6 and abs(oy - lat) < r + R_ROBOT + 0.2:
            return 3 if oy > lat else 2        # move away from the obstacle
    if lat - lane_y > 0.3:
        return 3
    if lat - lane_y < -0.3:
        return 2
    return 0


def make_episode(seed):
    rng = np.random.default_rng(seed)
    style = {"sky": rng.integers(120, 230, 3), "grass": rng.integers(30, 120, 3) * [0.6, 1.2, 0.6],
             "road": rng.integers(40, 110, 1).repeat(3), "dashed": bool(rng.integers(2))}
    style["grass"] = np.clip(style["grass"], 0, 255).astype(np.uint8)
    lane_y = rng.choice([-1.0, 1.0])
    lat = lane_y + rng.uniform(-0.4, 0.4)
    x = 0.0
    obstacles = [(rng.uniform(4, 9) + 7 * i, rng.uniform(-1.5, 1.5), rng.uniform(0.2, 0.6),
                  rng.integers(0, 256, 3)) for i in range(rng.integers(1, 4))]
    images, labels, executed, states = [], [], [], []
    for _ in range(STEPS):
        rel = [(ox - x, oy, r, c) for ox, oy, r, c in obstacles]
        a = expert(lat, rel, lane_y)
        e = a if rng.random() > 0.2 else int(rng.choice([0, 2, 3]))
        images.append(render(rng, lat, rel, style))
        labels.append(a); executed.append(e); states.append((x, lat, 0.0))
        dx, dy = {0: (STEP, 0), 1: (-STEP, 0), 2: (0, STEP), 3: (0, -STEP)}[e]
        x, lat = x + dx, float(np.clip(lat + dy, -3.5 + R_ROBOT, 3.5 - R_ROBOT))
    return dict(images=np.stack(images), expert_action=np.array(labels, np.int8),
                executed_action=np.array(executed, np.int8), state=np.array(states, np.float32))


DATA = WORK / "synthetic_v0"
t = time.time()
for split, seeds in {"train": range(0, int(N_EPISODES * 0.8)),
                     "val": range(int(N_EPISODES * 0.8), N_EPISODES)}.items():
    (DATA / split).mkdir(parents=True, exist_ok=True)
    for s in seeds:
        np.savez_compressed(DATA / split / f"ep_{s:06d}.npz", **make_episode(s))
labels = np.concatenate([np.load(f)["expert_action"] for f in sorted(DATA.glob("*/*.npz"))])
dist = {ACTIONS[i]: int((labels == i).sum()) for i in range(4)}
checks["dataset"] = (True, f"{N_EPISODES} episodes x {STEPS} steps in {time.time()-t:.1f}s, labels {dist}")
print(checks["dataset"])

In [ ]:
# Look at a few frames with their expert labels
import matplotlib.pyplot as plt
ep = np.load(sorted((DATA / "train").glob("*.npz"))[0])
fig, axes = plt.subplots(2, 5, figsize=(15, 4.5))
for ax, i in zip(axes.flat, np.linspace(0, STEPS - 1, 10).astype(int)):
    ax.imshow(ep["images"][i]); ax.axis("off")
    ax.set_title(f"step {i}: {ACTIONS[ep['expert_action'][i]]}")
plt.tight_layout(); plt.show()

## 5. Zip → Drive → back

The Colab FAQ advises against many small reads from the mounted Drive. The
plan is to keep datasets on Drive as one zip, and copy and unzip them on the VM.

In [ ]:
import shutil, zipfile
t = time.time()
zip_local = shutil.make_archive(str(WORK / "synthetic_v0"), "zip", DATA)
zip_drive = BASE / "datasets" / "synthetic_v0.zip"
shutil.copy(zip_local, zip_drive)
t_up = time.time() - t
shutil.rmtree(DATA)
t = time.time()
shutil.copy(zip_drive, WORK / "from_drive.zip")
with zipfile.ZipFile(WORK / "from_drive.zip") as z:
    z.extractall(DATA)
t_down = time.time() - t
n = len(list(DATA.glob("*/*.npz")))
size = zip_drive.stat().st_size / 2**20
checks["drive_roundtrip"] = (n == N_EPISODES, f"{size:.1f} MiB zip, to Drive {t_up:.1f}s, back+unzip {t_down:.1f}s, {n} episodes")
print(checks["drive_roundtrip"])

## 6. Training

Everything is loaded into RAM at the start (the real 40k-frame dataset at
160×120 is about 2.3 GB, which also fits). Pretrained ResNet18 with a 4-way
head, class-weighted cross-entropy. The checkpoint is written to Drive after
every epoch.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def load_split(split):
    files = sorted((DATA / split).glob("*.npz"))
    imgs = np.concatenate([np.load(f)["images"] for f in files])
    ys = np.concatenate([np.load(f)["expert_action"] for f in files]).astype(np.int64)
    return torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(ys)


Xtr, Ytr = load_split("train")
Xva, Yva = load_split("val")
print("train", tuple(Xtr.shape), "val", tuple(Xva.shape))

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Policy(nn.Module):
    """uint8 image batch [B,3,H,W] -> action logits [B,4]. Normalization is inside so the export is self-contained."""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
        self.net = torchvision.models.resnet18(weights=weights)
        self.net.fc = nn.Linear(self.net.fc.in_features, 4)
        self.register_buffer("mean", MEAN.clone())
        self.register_buffer("std", STD.clone())

    def forward(self, x):
        return self.net((x.float() / 255 - self.mean) / self.std)


counts = torch.bincount(Ytr, minlength=4).float()
class_w = (counts.sum() / (4 * counts.clamp(min=1))).to(DEVICE)
CKPT = RUN_DIR / "last.pt"


def train(epochs, resume=True, bs=64):
    model = Policy().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    start = 0
    if resume and CKPT.exists():
        ck = torch.load(CKPT, map_location=DEVICE)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start = ck["epoch"]
        print(f"resumed from epoch {start}")
    for epoch in range(start, epochs):
        model.train(); t = time.time(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xtr[idx].to(DEVICE)), Ytr[idx].to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); opt.step()
        acc, recall = evaluate(model)
        print(f"epoch {epoch+1}/{epochs}  loss {loss.item():.3f}  val acc {acc:.3f}  "
              f"recall {recall}  ({time.time()-t:.1f}s)")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch + 1}, CKPT)
    return model


@torch.no_grad()
def evaluate(model, bs=256):
    model.eval()
    pred = torch.cat([model(Xva[i:i + bs].to(DEVICE)).argmax(1).cpu() for i in range(0, len(Xva), bs)])
    recall = {ACTIONS[c]: round(float((pred[Yva == c] == c).float().mean()), 2)
              for c in range(4) if (Yva == c).any()}
    return float((pred == Yva).float().mean()), recall


if CKPT.exists():
    CKPT.unlink()   # start fresh each time the notebook runs
model = train(EPOCHS - 1)
acc, _ = evaluate(model)
baseline = float(torch.bincount(Yva, minlength=4).max()) / len(Yva)   # always predict the most common action
checks["training"] = (acc > baseline + 0.1, f"val acc {acc:.3f} after {EPOCHS-1} epochs, majority-class baseline {baseline:.3f}")
print(checks["training"])

## 7. Resume from the Drive checkpoint

Simulates a disconnect: a fresh model is built and continues from the saved epoch.

In [ ]:
del model
model = train(EPOCHS, resume=True)
ck = torch.load(CKPT, map_location="cpu")
checks["resume"] = (ck["epoch"] == EPOCHS, f"checkpoint at epoch {ck['epoch']} in {CKPT}")
print(checks["resume"])

## 8. TorchScript export

`drive.py` will load this file in Isaac Sim without needing the model's Python code.

In [ ]:
model.eval()
example = Xva[:8].to(DEVICE)
ts_path = RUN_DIR / "model.ts"
torch.jit.trace(model, example).save(str(ts_path))
loaded = torch.jit.load(str(ts_path), map_location=DEVICE)
with torch.no_grad():
    diff = (loaded(example) - model(example)).abs().max().item()
checks["export"] = (diff < 1e-4, f"{ts_path.name} {ts_path.stat().st_size/2**20:.1f} MiB, max diff {diff:.2e}")
print(checks["export"])

## Summary

In [ ]:
print(f"{'check':16} result  detail")
for name, (ok, detail) in checks.items():
    print(f"{name:16} {'PASS' if ok else 'FAIL':6}  {detail}")
print("\nAll good." if all(ok for ok, _ in checks.values()) else "\nSome checks failed - see above.")